# Seto-1B Training

Tiny language model (~1B params) for mobile deployment.

## Setup
1. Add this notebook to a Kaggle dataset containing the `seto/` code
2. Add training data via **Add Data** (FineWeb-Edu, SlimPajama, etc.)
3. Set dataset paths in the config below
4. Run All

## Specs
- 22 layers, d_model=2048, GQA (16Q/4KV), SwiGLU, RoPE
- ~1.04B params → ~520MB after INT4 quantization
- Optimized for 2x T4 GPU (16GB each)
- Checkpoints saved as ZIP every 1000 steps

In [ ]:
# Install dependencies
!pip install -q tokenizers datasets accelerate tqdm

In [ ]:
# Check GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"    Memory: {torch.cuda.get_device_properties(i).total_mem / 1e9:.1f} GB")
    print(f"bf16 supported: {torch.cuda.is_bf16_supported()}")

In [ ]:
# Setup code path - copy seto to working directory
import shutil
import os

# If seto is uploaded as a Kaggle dataset, copy it
SETO_DATASET_PATH = "/kaggle/input/seto-model"  # <-- adjust if different name
WORKING_DIR = "/kaggle/working"

if os.path.exists(f"{SETO_DATASET_PATH}/seto"):
    if not os.path.exists(f"{WORKING_DIR}/seto"):
        shutil.copytree(f"{SETO_DATASET_PATH}/seto", f"{WORKING_DIR}/seto")
        print(f"Copied seto to {WORKING_DIR}/seto")
    else:
        print("seto already in working dir")
elif os.path.exists("/kaggle/working/seto"):
    print("seto already in working dir")
else:
    print(f"WARNING: Seto code not found at {SETO_DATASET_PATH}")
    print("Upload seto/ folder as a Kaggle dataset and set SETO_DATASET_PATH")

os.chdir(f"{WORKING_DIR}")
print(f"Working dir: {os.getcwd()}")
print(f"Contents: {os.listdir('.')}")

In [ ]:
# Find training datasets in /kaggle/input
import glob

KAGGLE_INPUT = "/kaggle/input"

# Auto-discover datasets
all_datasets = []
for d in sorted(os.listdir(KAGGLE_INPUT)):
    full_path = os.path.join(KAGGLE_INPUT, d)
    if os.path.isdir(full_path):
        # Look for data files
        jsonl_files = glob.glob(f"{full_path}/**/*.jsonl", recursive=True)
        json_files = glob.glob(f"{full_path}/**/*.json", recursive=True)
        parquet_files = glob.glob(f"{full_path}/**/*.parquet", recursive=True)
        text_files = glob.glob(f"{full_path}/**/*.txt", recursive=True)
        
        total_files = len(jsonl_files) + len(json_files) + len(parquet_files) + len(text_files)
        if total_files > 0:
            all_datasets.append({
                "name": d,
                "path": full_path,
                "jsonl": len(jsonl_files),
                "json": len(json_files),
                "parquet": len(parquet_files),
                "txt": len(text_files),
            })

print(f"Found {len(all_datasets)} datasets with data files:")
for ds in all_datasets:
    print(f"  {ds['name']}: {ds['jsonl']} jsonl, {ds['json']} json, {ds['parquet']} parquet, {ds['txt']} txt")

# ============================================================
# CONFIGURE: Set your training dataset path here
# ============================================================
TRAIN_DATASET = all_datasets[0]["path"] if all_datasets else None
print(f"\nUsing dataset: {TRAIN_DATASET}")

In [ ]:
# Setup sys.path and import Seto
import sys
sys.path.insert(0, "/kaggle/working")

from seto.config import ModelConfig, TrainingConfig
from seto.model import SetoLM
from seto.tokenizer import SetoTokenizer
from seto.data import StreamingDataset, find_datasets

print("Seto imported successfully")

In [ ]:
# ============================================================
# TRAINING CONFIGURATION
# ============================================================

model_config = ModelConfig(
    vocab_size=32000,
    d_model=2048,
    n_layers=22,
    n_heads=16,
    n_kv_heads=4,
    d_ff=5504,
    max_seq_len=2048,
    dropout=0.0,
    tie_embeddings=True,
)

train_config = TrainingConfig(
    # Optimization
    lr=3e-4,
    min_lr=3e-5,
    weight_decay=0.1,
    beta1=0.9,
    beta2=0.95,
    max_grad_norm=1.0,
    
    # Schedule
    warmup_steps=2000,
    max_steps=50000,  # ~3-4 hours on 2xT4
    
    # Batch — tuned for 2x T4 (16GB each)
    batch_size=4,          # per GPU
    grad_accum_steps=8,    # effective batch = 4*8*2 = 64
    
    # Precision
    use_bf16=True,
    use_gradient_checkpointing=True,
    
    # Checkpointing
    save_every=1000,
    eval_every=500,
    keep_last_n=3,
    checkpoint_dir="/kaggle/working/checkpoints",
    zip_checkpoints=True,
    
    # Data
    train_datasets=[TRAIN_DATASET] if TRAIN_DATASET else [],
    num_workers=2,
)

print(f"Model params: ~{model_config.num_params():,}")
print(f"Effective batch size: {train_config.effective_batch_size}")
print(f"Tokens per step: ~{train_config.tokens_per_step:,}")
print(f"Total tokens over {train_config.max_steps} steps: ~{train_config.max_steps * train_config.tokens_per_step:,}")

In [ ]:
# Train tokenizer on the data
from seto.tokenizer import SetoTokenizer

TOKENIZER_DIR = "/kaggle/working/seto-tokenizer"

if os.path.exists(f"{TOKENIZER_DIR}/tokenizer.json"):
    print("Tokenizer already trained, loading...")
    tokenizer = SetoTokenizer.from_pretrained(TOKENIZER_DIR)
else:
    print("Training tokenizer...")
    
    # Collect text files for tokenizer training
    train_files = []
    if TRAIN_DATASET:
        for ext in ["*.jsonl", "*.txt", "*.json"]:
            train_files.extend(glob.glob(f"{TRAIN_DATASET}/**/{ext}", recursive=True))
    
    if not train_files:
        print("ERROR: No training files found for tokenizer")
    else:
        print(f"Training on {len(train_files)} files")
        tokenizer = SetoTokenizer(vocab_size=model_config.vocab_size)
        tokenizer.train(train_files[:10], TOKENIZER_DIR)  # use first 10 files
        print(f"Tokenizer saved to {TOKENIZER_DIR}")

print(f"Vocab size: {len(tokenizer)}")
print(f"Special tokens: {tokenizer.special_tokens}")

In [ ]:
# Initialize model
model = SetoLM(model_config)
print(f"Model initialized: {model.count_parameters():,} parameters")
print(f"Model size: {sum(p.numel() * p.element_size() for p in model.parameters()) / 1e9:.2f} GB")

In [ ]:
# ============================================================
# MULTI-GPU TRAINING (2x T4)
# ============================================================
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP
from seto.trainer import SetoTrainer

NUM_GPUS = torch.cuda.device_count()
print(f"Training on {NUM_GPUS} GPU(s)")

if NUM_GPUS >= 2:
    # Multi-GPU: use torchrun or spawn
    print("Multi-GPU training: launching via torchrun")
    print("Run this in terminal:")
    print(f"  torchrun --nproc_per_node={NUM_GPUS} --master_port=29500 seto/scripts/train.py \\")
    print(f"    --data-dir {TRAIN_DATASET} \\")
    print(f"    --output-dir /kaggle/working/seto-output \\")
    print(f"    --tokenizer {TOKENIZER_DIR} \\")
    print(f"    --batch-size {train_config.batch_size} \\")
    print(f"    --max-steps {train_config.max_steps}")
    
    # For notebook: launch training as subprocess
    import subprocess
    cmd = [
        "torchrun",
        f"--nproc_per_node={NUM_GPUS}",
        "--master_port=29500",
        "seto/scripts/train.py",
        "--data-dir", TRAIN_DATASET,
        "--output-dir", "/kaggle/working/seto-output",
        "--tokenizer", TOKENIZER_DIR,
        "--batch-size", str(train_config.batch_size),
        "--max-steps", str(train_config.max_steps),
    ]
    print(f"\nLaunching: {' '.join(cmd)}")
    result = subprocess.run(cmd, cwd="/kaggle/working")
    print(f"Training finished with return code: {result.returncode}")

elif NUM_GPUS == 1:
    # Single GPU
    print("Single GPU training")
    train_config.local_rank = -1
    train_config.world_size = 1
    train_config.batch_size = 8  # can use larger batch on single GPU
    
    dataset = StreamingDataset(TRAIN_DATASET, seq_len=model_config.max_seq_len)
    trainer = SetoTrainer(
        model=model,
        train_dataset=dataset,
        config=train_config,
        local_rank=-1,
    )
    trainer.train()

else:
    print("WARNING: No GPU detected. Training on CPU (very slow)")
    train_config.use_bf16 = False
    train_config.batch_size = 2
    train_config.grad_accum_steps = 16
    
    dataset = StreamingDataset(TRAIN_DATASET, seq_len=model_config.max_seq_len)
    trainer = SetoTrainer(
        model=model,
        train_dataset=dataset,
        config=train_config,
        local_rank=-1,
    )
    trainer.train()

In [ ]:
# ============================================================
# SAVE FINAL MODEL & CHECKPOINTS
# ============================================================
import zipfile

OUTPUT_DIR = "/kaggle/working/seto-output"
FINAL_DIR = "/kaggle/working/seto-final"

os.makedirs(FINAL_DIR, exist_ok=True)

# Save final model weights
state_dict = model.module.state_dict() if hasattr(model, "module") else model.state_dict()
torch.save(state_dict, f"{FINAL_DIR}/model.pt")

# Save config
import json
with open(f"{FINAL_DIR}/config.json", "w") as f:
    json.dump({
        "model": model_config.__dict__,
        "training": {k: str(v) for k, v in train_config.__dict__.items()},
    }, f, indent=2)

# Copy tokenizer
if os.path.exists(TOKENIZER_DIR):
    shutil.copytree(TOKENIZER_DIR, f"{FINAL_DIR}/tokenizer", dirs_exist_ok=True)

# Zip everything
ZIP_PATH = "/kaggle/working/seto-final.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(FINAL_DIR):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, FINAL_DIR)
            zf.write(file_path, arcname)

print(f"Final model saved: {ZIP_PATH}")
print(f"Zip size: {os.path.getsize(ZIP_PATH) / 1e6:.1f} MB")

In [ ]:
# List all checkpoints
CKPT_DIR = "/kaggle/working/checkpoints"
if os.path.exists(CKPT_DIR):
    zips = sorted([f for f in os.listdir(CKPT_DIR) if f.endswith(".zip")])
    print(f"Checkpoints ({len(zips)}):")
    for z in zips:
        size = os.path.getsize(os.path.join(CKPT_DIR, z)) / 1e6
        print(f"  {z} ({size:.1f} MB)")
    
    # Zip all checkpoints into one archive for easy download
    ALL_CKPT_ZIP = "/kaggle/working/seto-all-checkpoints.zip"
    with zipfile.ZipFile(ALL_CKPT_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
        for z in zips:
            zf.write(os.path.join(CKPT_DIR, z), z)
    print(f"\nAll checkpoints zipped: {os.path.getsize(ALL_CKPT_ZIP) / 1e6:.1f} MB")
else:
    print("No checkpoints found")

# List working directory
print(f"\n/kaggle/working contents:")
for f in sorted(os.listdir("/kaggle/working")):
    if os.path.isfile(f"/kaggle/working/{f}"):
        size = os.path.getsize(f"/kaggle/working/{f}") / 1e6
        print(f"  {f} ({size:.1f} MB)")

In [ ]:
# Quick inference test (before quantization)
model.eval()
device = next(model.parameters()).device

test_prompt = "<|system|>\nYou are Seto, a helpful assistant.\n<|user|>\nWhat is 2+2?\n<|assistant|>\n"
input_ids = torch.tensor([tokenizer.encode(test_prompt, add_bos=True, add_eos=False)], device=device)

with torch.no_grad():
    for _ in range(100):
        logits, _ = model(input_ids)
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        input_ids = torch.cat([input_ids, next_token], dim=-1)

output = tokenizer.decode(input_ids[0].tolist(), skip_special_tokens=True)
print("Inference test:")
print(output)